# Лабораторная работа №3

## Контекстный перевод и локализация с помощью LLM (GigaChat / Yandex AI Studio / локальная LLM)

**Студент:** ____________________ **Группа:** __________
**Вариант:** ______ (**тот же, что в ЛР № 1 и ЛР № 2**; предметная область: _______________________)
**Основная модель:** ____________ (GigaChat / Yandex AI Studio / локальная через Ollama)

**Раздел курса:** 5 · **Максимум:** 20 баллов · **Среда:** Google Colab (для локальной модели — GPU T4)

### Порядок выполнения

1. Прочитайте разделы «Суть работы», «Связь с ЛР № 1–2» и «Теоретическое обоснование».
2. Блок 1 — выберите `PROVIDER`. Ключи облачных моделей добавьте в **Colab Secrets** (значок ключа
   слева), а не в код.
3. Блок 2 — загрузите `variant_<N>.tmx` и `variant_<N>.tbx` из ЛР № 1 и (желательно)
   `lab02_results.zip` из ЛР № 2.
4. Выполните блоки по порядку, заполняя места `TODO`. Автотесты в конце проверяют функции без
   вызова LLM.
5. Заполните вручную колонки `confirmed` в `terminology_qa.csv` и `preferred`/`comment` в
   `comparison.csv`, допишите выводы.

> **Режим `mock` к сдаче не принимается.** Он нужен только для проверки кода без модели и
> возвращает эталонный перевод, а не ответ LLM.

## Суть работы за пять минут

### Что здесь происходит

В ЛР № 1 вы собрали ресурсы переводчика — терминологическую базу (TBX) и память переводов (TMX).
В ЛР № 2 перевели те же сегменты нейросетевыми моделями MarianMT и NLLB и увидели главную
проблему: гладкий текст при неверных терминах. Теперь тот же материал переводит **большая
языковая модель (LLM)**, но уже не «как есть», а по **управляемому сценарию**: с системной
инструкцией, утверждённой терминологией, примерами и строгим форматом ответа.

Работа отвечает на практический вопрос: **насколько можно управлять переводом, если модели
можно объяснить задачу словами — и где это управление перестаёт работать?**

### Что вы исследуете

| Вопрос | Как отвечаем | Блок |
|---|---|---|
| Помогает ли модели инструкция и глоссарий? | сравнение трёх режимов: zero-shot → controlled → few-shot | 7, 8 |
| Можно ли получить ответ, пригодный для автоматической обработки? | JSON Schema + проверка Pydantic | 5, 7 |
| Сохраняет ли модель разметку, числа, код и модальность? | проверочные случаи из ваших сегментов | 9 |
| Можно ли точечно исправить термины, не переписав перевод? | targeted repair и контроль изменений | 10 |
| Управляется ли стиль? | академический стиль против инструктивного | 11 |
| Что лучше для вашего материала — NMT или LLM? | сводная таблица с результатами ЛР № 2 | 8, 12 |

### Главная мысль, к которой ведёт работа

LLM отличается от NMT не «качеством вообще», а **управляемостью**: ей можно передать
терминологию, стиль и формат. Но инструкция — не гарантия. Поэтому пайплайн строится как
цепочка «сгенерировать → проверить → точечно исправить → проверить снова», а решение о
принятии перевода остаётся за человеком.

## Связь с ЛР № 1–2 и место работы в курсе

| Источник | Файл | Что из него берётся в ЛР № 3 |
|---|---|---|
| ЛР № 1 | `variant_<N>.tmx` | тестовые сегменты, эталонный перевод, пул примеров few-shot |
| ЛР № 1 | `variant_<N>.tbx` | терминологические ограничения в промпте и автоматическая QA |
| ЛР № 2 | `results_marian.jsonl`, `results_nllb.jsonl` | точка сравнения NMT и LLM на тех же сегментах |
| ЛР № 2 | `glossary_qa.csv` | список терминов, на которых NMT ошиблась: их проверяем в первую очередь |

Разбор TMX и TBX выполняет тот же адаптер, что и в ЛР № 2 (блок 2.0): `tuid` → `segment_id`,
английский `<seg>` → `source`, русский `<seg>` → `reference`, `<term>` en/ru → глоссарий.

```
ЛР № 1  TMX + TBX            ЛР № 2  NMT (Marian, NLLB)        ЛР № 3  LLM-пайплайн             ЛР № 4
повторное использование  ─►  перевод без управления      ─►  перевод с инструкцией,      ─►  метрики качества
перевода                     терминология «как выйдет»       терминами, форматом, repair      BLEU, chrF, COMET
```

Метрики качества (BLEU, chrF, COMET) в этой работе сознательно **не** считаются — это тема ЛР № 4.
Здесь качество оценивается по соблюдению глоссария, сохранению разметки и экспертному разбору.

## Теоретическое обоснование

### 1. Чем LLM-перевод отличается от NMT

NMT-модели из ЛР № 2 (MarianMT, NLLB) — **специализированные** энкодер-декодеры: их обучали только
переводу, и на вход они принимают только исходный текст. LLM — **универсальная** генеративная
модель (обычно только декодер), обученная продолжать текст и выполнять инструкции. Отсюда три
практических следствия:

* LLM можно передать **контекст**: терминологию, стиль, назначение текста, примеры решений;
* LLM может вернуть ответ в **заданной структуре** (JSON), а не только строку перевода;
* LLM склонна к ошибкам, которых NMT почти не делает: **добавлению** фактов, **пропуску**
  частей, **перефразированию** сверх задачи и выходу за формат.

Исследования показывают, что крупные LLM конкурентоспособны с NMT на распространённых языковых
парах, но их качество сильно зависит от формулировки запроса и выбора примеров (Vilar et al.,
2023; Hendy et al., 2023; Zhu et al., 2024).

### 2. Из чего состоит запрос

| Часть | Назначение | Где в работе |
|---|---|---|
| **Системная инструкция** | роль, задача, правила, формат | `SYSTEM_TEMPLATE`, блок 4 |
| **Терминологический контекст** | обязательные эквиваленты для терминов **этого** сегмента | `relevant_terms()`, блок 4 |
| **Примеры (few-shot)** | образцы решений в формате «запрос → ответ» | `select_fewshot()`, блок 3 |
| **Пользовательское сообщение** | сегмент в разделителях `<<< >>>` и его идентификатор | `user_payload()`, блок 4 |

**Почему в промпт идут только термины сегмента, а не весь глоссарий.** Глоссарий студента —
100+ терминов. Если передавать его целиком, запрос удлиняется в десятки раз, растёт стоимость,
а внимание модели распределяется по нерелевантным строкам. Подход «словарь на уровне фразы»
(Ghazvininejad et al., 2023) передаёт только те пары, которые встречаются в переводимом тексте.

### 3. Zero-shot, controlled, few-shot

* **Zero-shot** — только «переведи». Базовая линия: так модель переводит «по умолчанию».
* **Controlled** — системная инструкция + термины сегмента. Проверяет, насколько модель следует
  явным ограничениям.
* **Few-shot** — то же плюс 2–4 примера. Модель учится по образцу прямо в запросе
  (*in-context learning*, Brown et al., 2020).

Два методических требования к примерам:

1. **Примеры не должны пересекаться с тестовыми сегментами.** Иначе модель просто «увидит
   ответ», и сравнение режимов потеряет смысл — это утечка данных. В коде стоит проверка.
2. **Пример показывает тип решения**, а не словарную пару: многозначность, составной термин,
   ложный друг переводчика, нормативная формулировка.

Связь с ЛР № 1: примеры можно подбирать из памяти переводов по сходству с сегментом — так же, как
работает нечёткий поиск (fuzzy match). Такой подход называют адаптивным переводом с LLM
(Moslem et al., 2023); в работе он включается параметром `FEWSHOT_STRATEGY = "fuzzy"`.

### 4. Структурированный вывод

JSON Schema описывает, какие поля и каких типов должны быть в ответе. Современные API умеют
**ограничивать генерацию** схемой: модель на каждом шаге может выбрать только токены, которые
не нарушают структуру (*constrained decoding*, Willard & Louf, 2023). У GigaChat это
`response_format` с типом `json_schema`, у Yandex AI Studio — `response_format={"json_schema": …}`,
у Ollama — параметр `format`.

**Зачем тогда Pydantic.** Ограничение формата гарантирует синтаксис, но не смысл: в поле
`translation` может оказаться пустая строка, текст на другом языке или чужой `segment_id`. Поэтому
проверка трёхуровневая: JSON разбирается → поля проверяются моделью Pydantic → проверяются
смысловые правила (язык, идентификатор, непустота). Невалидный ответ повторно запрашивается с
указанием ошибки, а число попыток сохраняется в трассировке.

### 5. Воспроизводимость

* **Температура 0** делает генерацию практически детерминированной; у GigaChat минимальное значение
  положительное, поэтому в коде стоит 0.01.
* **Фиксируется точная версия модели.** Облачные модели обновляются без предупреждения, и тот же
  запрос через месяц может дать другой ответ. У Ollama версия — это `digest` сборки, у облачных
  API — поле версии в ответе.
* **Промпты версионируются**: номер версии плюс хэш текста каждого промпта. Изменили инструкцию —
  изменился хэш, и результаты уже нельзя сравнивать с прошлыми напрямую.

### 6. Терминологический контроль и repair

Автоматическая проверка ищет русский эквивалент термина в переводе с учётом словоизменения.
Упрощённая нормализация по окончаниям неизбежно даёт **ложные срабатывания** (синоним,
перестройка предложения) и **пропуски** — поэтому каждое срабатывание подтверждается вручную.

Исправление выполняется **узким** запросом: «замени только эти термины, остальное не трогай».
Контроль двойной: после repair заново считается число нарушений, а доля неизменённого текста
(`unchanged_ratio`, отношение сходства строк до и после) показывает, не переписала ли модель
перевод целиком. Значение около 0.9 — точечная правка, ниже 0.7 — фактически новый перевод.

Альтернатива — простая замена строк (`str.replace`) — не работает для русского языка: термин
нужно согласовать по падежу и числу с окружением. Поэтому в работе замена поручается модели,
а проверка — коду.

### 7. Разметка, числа, код и модальность

В учебном тексте часть элементов переводить **нельзя**: фрагменты кода, пути, идентификаторы,
числа и единицы. Другие нужно **сохранить** по структуре: выделение, списки. Модальность
(*must* — «должен», *should* — «следует», *may* — «может») определяет обязательность действия, и её
потеря в инструкции меняет смысл требования. Исходные сегменты — обычно простой текст, поэтому
проверочные случаи строятся из них: выделение термина, список из двух сегментов, фрагмент кода.

### 8. Безопасность и данные

Текст, отправленный в облачный API, покидает вашу среду. Для учебного модуля это допустимо, но
персональные данные (152-ФЗ) и закрытые материалы в облачные модели не отправляются — для них
предназначен локальный вариант. Ключи хранятся в Colab Secrets или `.env`, который исключён из Git.

## Блок 0. Установка зависимостей

**Зачем.** Базовые библиотеки: `pydantic` — проверка структуры ответа, `python-dotenv` — чтение
ключей из `.env` вне Colab, `pandas` — таблицы. Библиотека конкретного провайдера (`ollama`,
`gigachat` или `yandex-ai-studio-sdk`) ставится в блоке 1 только для выбранного варианта.

In [ ]:
!pip install -q "pydantic>=2" python-dotenv pandas

## Блок 1. Параметры эксперимента и секреты

**Зачем.** Все решения, от которых зависит результат, собраны в одном месте: провайдер, точная
модель, температура, зерно, версия промптов, число сегментов. Именно эти значения попадут в
трассировку каждого вызова и в отчёт.

**Как подключить облачную модель.**

| Провайдер | Что нужно | Имена секретов |
|---|---|---|
| Ollama (локально) | GPU T4 в Colab, ключи не нужны | — |
| GigaChat | ключ авторизации из личного кабинета Studio; для физических лиц scope `GIGACHAT_API_PERS` | `GIGACHAT_CREDENTIALS`, `GIGACHAT_SCOPE` |
| Yandex AI Studio | каталог и API-ключ сервисного аккаунта с ролью `ai.languageModels.user` | `YANDEX_FOLDER_ID`, `YANDEX_API_KEY` |

Секреты добавляются в **Colab Secrets** (значок ключа на левой панели) и читаются функцией
`get_secret()`; их значения нигде не печатаются.

> GigaChat использует сертификаты НУЦ Минцифры. В учебной среде допустимо
> `GIGACHAT_VERIFY_SSL=false`; в рабочей — установите корневой сертификат.

In [ ]:
from __future__ import annotations

import difflib
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

# ------------------------------------------------------------------ параметры эксперимента
PROVIDER = "ollama"          # "ollama" | "gigachat" | "yandex" | "mock" (mock — только проверка кода)
MODELS = {
    "ollama": "qwen2.5:7b",          # локальная instruct-модель, работает на T4 без ключей
    "gigachat": "GigaChat-2-Max",    # облако Сбера, нужен ключ авторизации
    "yandex": "yandexgpt",           # Yandex AI Studio, нужны folder_id и API-ключ
    "mock": "mock-echo",             # НЕ модель: эхо эталона для проверки пайплайна
}
MODEL = MODELS[PROVIDER]
TEMPERATURE = 0.0            # воспроизводимость важнее разнообразия
SEED = 42
MAX_TOKENS = 800
PROMPT_VERSION = "v1.0"

N_TEST = 12                  # одинаковые сегменты во всех режимах
FEWSHOT_K = 2                # примеров в few-shot (2–4)
FEWSHOT_STRATEGY = "static"  # "static" — отобранные вручную, "fuzzy" — ближайшие по сходству из ТМ
N_STYLE = 3                  # сегментов для сравнения стилей
N_REPAIR_MIN = 3             # минимум случаев targeted repair
MIN_SEGMENTS = N_TEST + FEWSHOT_K   # тест + пул few-shot без пересечения
MIN_TERMS = 10

BASE = Path("/content/lab03") if Path("/content").exists() else Path("lab03_work")
DATA, RESULTS, PROMPTS, FIGURES = BASE / "data", BASE / "results", BASE / "prompts", BASE / "figures"
for folder in (DATA, RESULTS, PROMPTS, FIGURES):
    folder.mkdir(parents=True, exist_ok=True)


def get_secret(name: str) -> Optional[str]:
    """Секрет из Colab Secrets или переменной окружения. Значение никогда не печатается."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    return os.environ.get(name)


def pip_install(*packages: str) -> None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=False)


PROVIDER_PACKAGES = {"ollama": ("ollama", "ollama"),
                     "gigachat": ("gigachat", "gigachat"),
                     "yandex": ("yandex-ai-studio-sdk", "yandex_ai_studio_sdk")}
if PROVIDER in PROVIDER_PACKAGES:
    package, module = PROVIDER_PACKAGES[PROVIDER]
    try:
        __import__(module)
    except ImportError:
        pip_install(package)

ENVIRONMENT = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "provider": PROVIDER,
    "model": MODEL,
    "temperature": TEMPERATURE,
    "seed": SEED,
    "prompt_version": PROMPT_VERSION,
    "base_dir": str(BASE),
}
for key, value in ENVIRONMENT.items():
    print(f"{key:>15}: {value}")
if PROVIDER == "mock":
    print("\n⚠ PROVIDER = mock: модель НЕ вызывается, результаты — эхо эталона. "
          "Режим нужен только для проверки кода и к сдаче не принимается.")

### Блок 1.1. Запуск локальной модели (Ollama)

**Зачем.** Локальная модель не требует ключей, не отправляет текст в облако и воспроизводится у
любого пользователя Colab. Ячейка ставит сервер Ollama, запускает его в фоне и скачивает модель
(`qwen2.5:7b` — около 4.7 ГБ, первая загрузка занимает 2–4 минуты).

**На что смотреть в выводе.** Строка с моделью в `ollama list` и её `digest` — это точная версия
сборки, которая попадёт в отчёт, и имя GPU. Если GPU не найден, подключите T4: на CPU генерация
идёт в десятки раз медленнее.

In [ ]:
# ============================================================================
#  Запуск локальной LLM через Ollama в Colab (только при PROVIDER = "ollama")
#  1) ставим сервер Ollama, 2) запускаем его в фоне, 3) скачиваем модель.
#  qwen2.5:7b весит около 4.7 ГБ и работает на T4; первая загрузка — 2–4 минуты.
# ============================================================================
OLLAMA_DIGEST = None
if PROVIDER == "ollama":
    if shutil.which("ollama") is None:
        subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null 2>&1", shell=True)
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL,
                              stderr=subprocess.DEVNULL)
    for _ in range(30):                       # ждём, пока сервер начнёт отвечать
        if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
            break
        time.sleep(1)
    subprocess.run(["ollama", "pull", MODEL], check=True)
    listing = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
    print(listing)
    for line in listing.splitlines():
        if line.startswith(MODEL):
            OLLAMA_DIGEST = line.split()[1]   # идентификатор конкретной сборки модели
    print("Модель:", MODEL, "| digest:", OLLAMA_DIGEST)
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    ENVIRONMENT.update({"model_digest": OLLAMA_DIGEST, "gpu": gpu or "нет"})
    print("GPU:", gpu or "не обнаружен — генерация будет идти на CPU медленно")
else:
    print(f"Ollama не требуется: PROVIDER = {PROVIDER}")

## Блок 2. Данные сквозного кейса

**Зачем.** Работа идёт на **тех же** данных, что ЛР № 1–2: сегменты и эталон — из вашей памяти
переводов, термины — из вашей терминологической базы. Результаты ЛР № 2 подключаются как
дополнительная точка сравнения NMT и LLM.

**Что делает код.** Кладёт в `DATA` артефакты ЛР № 1 (у эталона они встроены), затем адаптер
(блок 2.0) разбирает TMX и TBX в таблицы `aligned_df` и `glossary_df`. Если рядом лежит архив
`lab02_results.zip`, из него читаются переводы MarianMT и NLLB.

**На что смотреть в выводе.** Число сегментов и терминов, предметная область из TMX и строку о
результатах ЛР № 2.

In [ ]:
# ============================================================================
#  Блок 2. Данные сквозного кейса: ваши артефакты ЛР № 1 и результаты ЛР № 2
# ============================================================================
VARIANT = None               # TODO: номер вашего варианта (тот же, что в ЛР № 1 и ЛР № 2)
VARIANT_DOMAIN = ""          # TODO: предметная область варианта

# Положите в DATA файлы variant_<N>.tmx и variant_<N>.tbx из ЛР № 1,
# а архив lab02_results.zip из ЛР № 2 — туда же (необязательно, но желательно).
# Способ 1:
# from google.colab import files
# for name, content in files.upload().items():
#     (DATA / name).write_bytes(content)
# Способ 2 (Google Drive):
# from google.colab import drive; drive.mount("/content/drive")
# import glob; [shutil.copy(p, DATA) for p in glob.glob("/content/drive/MyDrive/lab01/variant_*.t*x")]

found = sorted(p.name for p in list(DATA.glob("*.tmx")) + list(DATA.glob("*.tbx")))
print("Найдено артефактов ЛР № 1:", found or "нет")

### Блок 2.0. Адаптер артефактов ЛР № 1 и результатов ЛР № 2

Тот же адаптер, что в ЛР № 2: `tmx_to_pairs()` и `tbx_to_glossary()` разбирают XML, `build_lab02_inputs()` собирает таблицы. Затем подключаются переводы MarianMT и NLLB из `lab02_results.zip`, если архив лежит в каталоге `DATA`.

In [ ]:
# --- как положить файлы ЛР № 1 в каталог DATA -------------------------------
# Способ 1 (Colab, вручную): выберите variant_<N>.tmx и variant_<N>.tbx
# from google.colab import files
# for name, content in files.upload().items():
#     (DATA / name).write_bytes(content)
#
# Способ 2 (Google Drive): файлы переживут перезапуск среды
# from google.colab import drive
# drive.mount("/content/drive")
# import shutil, glob
# for path in glob.glob("/content/drive/MyDrive/lab01/variant_*.t*x"):
#     shutil.copy(path, DATA)



# ============================================================================
#  Блок 2.0. Адаптер: артефакты ЛР № 1 -> входные файлы ЛР № 2
#  Читает variant_<N>.tmx и variant_<N>.tbx (либо CSV-выгрузки для Smartcat)
#  и собирает три файла в DATA: segments_en.csv, glossary.csv, aligned_reference.csv
# ============================================================================
import glob
import re
import xml.etree.ElementTree as _ET

try:                                    # lxml безопаснее и быстрее, но не обязателен
    from lxml import etree as _LET

    def _parse_xml(path):
        parser = _LET.XMLParser(resolve_entities=False, load_dtd=False, no_network=True)
        return _LET.parse(str(path), parser)
except ImportError:                     # pragma: no cover
    def _parse_xml(path):
        return _ET.parse(str(path))

XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"
MIN_SEGMENTS = globals().get("MIN_SEGMENTS", 14)
MIN_TERMS = globals().get("MIN_TERMS", 10)
LAB1_DIR = DATA          # положите сюда файлы ЛР № 1 (.tmx/.tbx или CSV-выгрузки)


def _lang_of(node) -> str:
    """Код языка из xml:lang (или из lang для старых файлов), без региона."""
    raw = node.get(XML_LANG) or node.get("lang") or ""
    return raw.split("-")[0].lower()


def tmx_to_pairs(path, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """TMX (память переводов ЛР № 1) -> таблица параллельных сегментов.

    Порядок <tu> в файле сохраняется: он задаёт порядок сегментов документа.
    Инлайновые теги TMX (ph/bpt/ept/it) игнорируются, берётся только текст.
    """
    root = _parse_xml(path).getroot()
    rows = []
    for order, tu in enumerate(root.iter("tu"), start=1):
        by_lang = {}
        for tuv in tu.findall("tuv"):
            seg = tuv.find("seg")
            if seg is None:
                continue
            text = re.sub(r"\s+", " ", "".join(seg.itertext())).strip()
            if text:
                by_lang[_lang_of(tuv)] = text
        if src_lang in by_lang and tgt_lang in by_lang:
            domain_node = tu.find("prop[@type='domain']")
            rows.append({"segment_id": f"s{order:03d}",
                         "tuid": tu.get("tuid", str(order)),
                         "source": by_lang[src_lang],
                         "reference": by_lang[tgt_lang],
                         "domain": (domain_node.text or "").strip()
                                   if domain_node is not None else ""})
    return pd.DataFrame(rows, columns=["segment_id", "tuid", "source", "reference", "domain"])


def tbx_to_glossary(path, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """TBX (терминологическая база ЛР № 1) -> таблица терминов en/ru.

    Учитываются оба способа записи термина: <tig><term> и <ntig><termGrp><term>,
    поэтому используется поиск по всем потомкам langSet.
    """
    root = _parse_xml(path).getroot()
    rows = []
    for entry in root.iter("termEntry"):
        by_lang = {}
        for lang_set in entry.findall("langSet"):
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if terms:
                by_lang[_lang_of(lang_set)] = terms[0]
        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({"en": by_lang[src_lang], "ru": by_lang[tgt_lang],
                         "entry_id": entry.get("id", "")})
    return pd.DataFrame(rows, columns=["en", "ru", "entry_id"])


def _first(pattern: str):
    """Первый файл, подходящий под шаблон (ищем и в LAB1_DIR, и в текущем каталоге)."""
    found = sorted(glob.glob(str(LAB1_DIR / pattern))) + sorted(glob.glob(pattern))
    return Path(found[0]) if found else None


def build_lab02_inputs(lab1_dir=None, src_lang: str = "en", tgt_lang: str = "ru") -> dict:
    """Собрать входные файлы ЛР № 2 из артефактов ЛР № 1.

    Приоритет источников:
      1) variant_*.tmx + variant_*.tbx  — основной путь (форматы CAT-систем);
      2) tm_segments_v*.csv + glossary_smartcat_v*.csv — выгрузки для Smartcat
         (колонки tuid/source_segment/target_segment и 'Source (en)'/'Target (ru)').

    Returns:
        Словарь с числом сегментов и терминов и именами использованных источников.
    """
    global LAB1_DIR
    if lab1_dir is not None:
        LAB1_DIR = Path(lab1_dir)

    used = {}
    tmx, tbx = _first("*.tmx"), _first("*.tbx")

    if tmx is not None:
        pairs = tmx_to_pairs(tmx, src_lang, tgt_lang)
        used["segments"] = tmx.name
    else:
        tm_csv = _first("tm_segments_v*.csv") or _first("tm_segments*.csv")
        if tm_csv is None:
            raise FileNotFoundError(
                "Не найдены ни variant_*.tmx, ни tm_segments_v*.csv. "
                "Положите артефакты ЛР № 1 в каталог DATA (см. инструкцию выше).")
        frame = pd.read_csv(tm_csv, encoding="utf-8-sig")
        pairs = pd.DataFrame({
            "segment_id": [f"s{i:03d}" for i in range(1, len(frame) + 1)],
            "tuid": frame.get("tuid", pd.Series(range(1, len(frame) + 1))).astype(str),
            "source": frame["source_segment"].astype(str).str.strip(),
            "reference": frame["target_segment"].astype(str).str.strip(),
            "domain": frame.get("domain", "")})
        used["segments"] = tm_csv.name

    if tbx is not None:
        terms = tbx_to_glossary(tbx, src_lang, tgt_lang)
        used["glossary"] = tbx.name
    else:
        gl_csv = _first("glossary_smartcat_v*.csv") or _first("glossary_smartcat*.csv")
        if gl_csv is None:
            raise FileNotFoundError(
                "Не найдены ни variant_*.tbx, ни glossary_smartcat_v*.csv.")
        frame = pd.read_csv(gl_csv, encoding="utf-8-sig")
        cols = {c.lower().strip(): c for c in frame.columns}
        en_col = cols.get("source (en)") or cols.get("en") or list(frame.columns)[0]
        ru_col = cols.get("target (ru)") or cols.get("ru") or list(frame.columns)[1]
        terms = pd.DataFrame({"en": frame[en_col].astype(str).str.strip(),
                              "ru": frame[ru_col].astype(str).str.strip(),
                              "entry_id": ""})
        used["glossary"] = gl_csv.name

    # чистка: пустые строки и дубликаты из ТМ и глоссария не нужны
    pairs = (pairs[(pairs["source"].str.len() > 0) & (pairs["reference"].str.len() > 0)]
             .drop_duplicates(subset=["source"]).reset_index(drop=True))
    pairs["segment_id"] = [f"s{i:03d}" for i in range(1, len(pairs) + 1)]
    terms = (terms[(terms["en"].str.len() > 0) & (terms["ru"].str.len() > 0)]
             .drop_duplicates(subset=["en"]).reset_index(drop=True))

    pairs[["segment_id", "source"]].to_csv(DATA / "segments_en.csv",
                                           index=False, encoding="utf-8")
    pairs[["segment_id", "source", "reference"]].to_csv(DATA / "aligned_reference.csv",
                                                        index=False, encoding="utf-8")
    terms[["en", "ru"]].to_csv(DATA / "glossary.csv", index=False, encoding="utf-8")

    domains = [d for d in pairs.get("domain", pd.Series(dtype=str)).fillna("").unique() if d]
    return {"segments": len(pairs), "terms": len(terms), "sources": used,
            "domain": domains[0] if domains else ""}


# --- запуск адаптера --------------------------------------------------------
try:
    info = build_lab02_inputs()
    print("Артефакты ЛР № 1 преобразованы во входные файлы ЛР № 2")
    print(f"  источник сегментов : {info['sources']['segments']}")
    print(f"  источник глоссария : {info['sources']['glossary']}")
    print(f"  сегментов          : {info['segments']}"
          f"   (минимум для сдачи: {MIN_SEGMENTS})")
    print(f"  терминов           : {info['terms']}"
          f"   (минимум для сдачи: {MIN_TERMS})")
    if info["domain"]:
        print(f"  предметная область : {info['domain']} (из вашей ЛР № 1)")
    if info["segments"] < MIN_SEGMENTS or info["terms"] < MIN_TERMS:
        print("\nОбъём меньше минимума ЛР № 3 (12 тестовых сегментов + пул few-shot). "
              "Расширьте PARALLEL_CORPUS в ноутбуке ЛР № 1 и перегенерируйте TMX.")
except FileNotFoundError as error:
    print("Адаптер пропущен:", error)
    print("Дальше будет использован демонстрационный набор — сдавать работу на нём нельзя.")


# --- рабочие таблицы ------------------------------------------------------------
if (DATA / "aligned_reference.csv").exists():
    aligned_df = pd.read_csv(DATA / "aligned_reference.csv")
    glossary_df = pd.read_csv(DATA / "glossary.csv")
else:                                             # мини-набор только для автотестов
    aligned_df = pd.DataFrame({
        "segment_id": ["s001", "s002", "s003", "s004"],
        "source": ["The flight controller stabilises the airframe during autonomous flight.",
                   "Before take-off the operator checks the telemetry link quality.",
                   "The operator uploads a new waypoint to the autopilot.",
                   "The avionics bay is cooled by a dedicated air duct."],
        "reference": ["Полётный контроллер стабилизирует планёр во время автономного полёта.",
                      "Перед взлётом оператор проверяет качество канала телеметрии.",
                      "Оператор загружает новую путевую точку в автопилот.",
                      "Отсек авионики охлаждается отдельным воздушным каналом."]})
    glossary_df = pd.DataFrame({"en": ["flight controller", "airframe", "telemetry link",
                                       "waypoint", "autopilot", "avionics"],
                                "ru": ["полётный контроллер", "планёр", "канал телеметрии",
                                       "путевая точка", "автопилот", "авионика"]})
    print("⚠ Артефакты ЛР № 1 не найдены: используется мини-набор (к сдаче не принимается)")

DOMAIN_EN = str(info.get("domain", "")) if "info" in dir() and isinstance(info, dict) else ""
REF_BY_ID = dict(zip(aligned_df["segment_id"], aligned_df["reference"]))
SRC_BY_ID = dict(zip(aligned_df["segment_id"], aligned_df["source"]))

# --- результаты ЛР № 2 (необязательная точка сравнения) ---------------------------
import zipfile
lab2_dir = DATA / "lab02"
zip_path = DATA / "lab02_results.zip"
if zip_path.exists() and not lab2_dir.exists():
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(lab2_dir)


def load_nmt(name: str) -> Dict[str, str]:
    hits = sorted(lab2_dir.rglob(name)) if lab2_dir.exists() else []
    if not hits:
        return {}
    rows = [json.loads(line) for line in hits[0].read_text(encoding="utf-8").splitlines() if line]
    return {row.get("id") or row.get("segment_id"): row.get("translation", "") for row in rows}


NMT = {"MarianMT": load_nmt("results_marian.jsonl"), "NLLB": load_nmt("results_nllb.jsonl")}
print(f"\nСегментов: {len(aligned_df)}, терминов: {len(glossary_df)}, домен: {DOMAIN_EN or '—'}")
print("Результаты ЛР № 2:", {k: len(v) for k, v in NMT.items()} if any(NMT.values())
      else "не найдены (сравнение с NMT будет пропущено)")

## Блок 3. Тестовые сегменты и примеры few-shot

**Зачем.** Три режима сравниваются только на **одинаковых** сегментах, а примеры few-shot берутся
из **других** сегментов памяти переводов. Если пример совпадёт с тестовым сегментом, модель
увидит готовый ответ — это утечка данных, и сравнение режимов станет бессмысленным.

**Что делает код.** Первые 12 сегментов — тестовые, остальные — пул примеров. Для каждого
примера записывается, **какой тип решения** он демонстрирует. Проверка `assert` останавливает
работу, если выборки пересеклись.

In [ ]:
# ============================================================================
#  Блок 3. Тестовые сегменты и примеры few-shot без пересечения
# ============================================================================
TEST_IDS = aligned_df["segment_id"].head(N_TEST).tolist()
POOL_IDS = [sid for sid in aligned_df["segment_id"] if sid not in TEST_IDS]

# TODO 3.1: выберите в POOL_IDS 2–4 примера и для каждого опишите ТИП переводческого решения
# (многозначность, составной термин, ложный друг переводчика, нормативная формулировка).
FEWSHOT_NOTES: Dict[str, str] = {
    # "s013": "...",
}


def words(text: str) -> set:
    return set(re.findall(r"[a-z]+", text.lower()))


def select_fewshot(source: str, k: int = FEWSHOT_K, strategy: str = FEWSHOT_STRATEGY) -> List[str]:
    if strategy == "fuzzy":
        scored = sorted(POOL_IDS, key=lambda sid: -len(words(source) & words(SRC_BY_ID[sid]))
                        / max(len(words(source) | words(SRC_BY_ID[sid])), 1))
        return scored[:k]
    curated = [sid for sid in FEWSHOT_NOTES if sid in POOL_IDS]
    return (curated + [sid for sid in POOL_IDS if sid not in curated])[:k]


assert not set(TEST_IDS) & set(POOL_IDS), "few-shot пересекается с тестом — утечка данных"
print(f"Тестовых сегментов: {len(TEST_IDS)}, пул few-shot: {len(POOL_IDS)}")

## Блок 4. Промпты и терминологический контекст

**Зачем.** Промпт — это «техническое задание» модели, и его нужно версионировать так же, как код.

**Что делает код.**

* задаёт системную инструкцию с доменом варианта, короткую zero-shot инструкцию, шаблон repair и
  два стилевых варианта;
* `relevant_terms()` выбирает из глоссария только термины, которые встречаются в сегменте;
* `user_payload()` собирает сообщение: идентификатор, термины, исходный текст в разделителях;
* сохраняет все промпты в `prompts/` и печатает их хэши.

**На что смотреть.** Пример пользовательского сообщения: в нём должны быть только термины этого
сегмента, а текст — внутри `<<< >>>`, чтобы модель не спутала инструкцию с переводимым текстом.

In [ ]:
# ============================================================================
#  Блок 4. Промпты: версии, хэши и терминологический контекст сегмента
# ============================================================================
# TODO 4.1: адаптируйте правила под свою предметную область (единицы, обозначения, стиль).
SYSTEM_TEMPLATE = """Ты — профессиональный локализатор учебных материалов в предметной области «{domain}».

Задача: переводить учебный текст с английского языка на русский для студентов вуза.

Правила:
1. Сохраняй смысл, модальность (must / should / can / may) и логические связи.
2. Термины из блока «Термины» передавай строго указанными русскими эквивалентами,
   согласуя их по падежу и числу.
3. Не переводи и не изменяй фрагменты в `обратных кавычках`, идентификаторы, пути к файлам,
   числа и единицы измерения.
4. Сохраняй Markdown-разметку: **выделение**, списки, заголовки.
5. Не добавляй факты, которых нет в исходнике; не сокращай текст.
6. Многозначные слова переводи в значении, принятом в предметной области.
7. {style}
8. Верни только JSON по заданной схеме: segment_id, translation, used_terms, warnings.
   В warnings укажи сомнительные места; если их нет — пустой список."""

STYLE_DEFAULT = "Пиши в нейтральном академическом стиле учебного пособия."
STYLE_A = "Пиши в академическом нейтральном стиле учебного пособия вуза: безличные и пассивные конструкции."
STYLE_B = ("Пиши как дружелюбный преподаватель, объясняющий первокурснику: обращение на «вы», "
           "побудительные формулировки, без разговорного сленга.")

ZERO_SYSTEM = ("Переведи текст с английского языка на русский. "
               "Верни только JSON по схеме: segment_id, translation, used_terms, warnings.")

REPAIR_TEMPLATE = """Исправь в переводе ТОЛЬКО перечисленные терминологические несоответствия.
Не переписывай остальные части перевода, не меняй порядок слов без необходимости,
не меняй код, Markdown, числа и факты. Согласуй исправленные термины по падежу и числу.

segment_id: {segment_id}
Исходный текст: {source}
Текущий перевод: {translation}
Нарушения (английский термин → обязательный русский эквивалент):
{violations}

Верни JSON по схеме; в warnings перечисли внесённые изменения."""

TERM_PATTERNS = {row.en: re.compile(r"(?<![\w-])" + re.escape(str(row.en)) + r"(?:s|es)?(?![\w-])",
                                    re.IGNORECASE)
                 for row in glossary_df.itertuples(index=False)}


def relevant_terms(source: str, glossary: pd.DataFrame = None) -> List[Tuple[str, str]]:
    """Термины глоссария, встречающиеся в сегменте (длинные — первыми)."""
    glossary = glossary_df if glossary is None else glossary
    # TODO 4.2: верните пары (en, ru) для терминов глоссария, которые встречаются в source.
    #   * поиск без учёта регистра и по границам слов: re.search(r"(?<![\w-])" + re.escape(en) + ...)
    #   * учтите множественное число (waypoint → waypoints);
    #   * отсортируйте по убыванию длины английского термина: «ground control station»
    #     должен идти раньше «station», если оба есть в глоссарии.
    return []


def user_payload(segment_id: str, source: str, with_terms: bool) -> str:
    lines = [f"segment_id: {segment_id}"]
    terms = relevant_terms(source) if with_terms else []
    if terms:
        lines.append("Термины (использовать строго):")
        lines += [f"- {en} = {ru}" for en, ru in terms]
    lines += ["Исходный текст:", "<<<", source, ">>>"]
    return "\n".join(lines)


def prompt_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:10]


SYSTEM_PROMPT = SYSTEM_TEMPLATE.format(domain=VARIANT_DOMAIN or DOMAIN_EN, style=STYLE_DEFAULT)
PROMPT_REGISTRY = {
    "system_controlled": SYSTEM_PROMPT,
    "system_zero": ZERO_SYSTEM,
    "system_style_A": SYSTEM_TEMPLATE.format(domain=VARIANT_DOMAIN or DOMAIN_EN, style=STYLE_A),
    "system_style_B": SYSTEM_TEMPLATE.format(domain=VARIANT_DOMAIN or DOMAIN_EN, style=STYLE_B),
    "repair": REPAIR_TEMPLATE,
}
for name, text in PROMPT_REGISTRY.items():
    (PROMPTS / f"{name}.txt").write_text(text, encoding="utf-8")
(PROMPTS / "prompt_system.txt").write_text(SYSTEM_PROMPT, encoding="utf-8")
(PROMPTS / "prompt_repair.txt").write_text(REPAIR_TEMPLATE, encoding="utf-8")
(PROMPTS / "fewshot.json").write_text(json.dumps(
    [{"segment_id": sid, "source": SRC_BY_ID[sid], "translation": REF_BY_ID[sid],
      "note": FEWSHOT_NOTES.get(sid, "")} for sid in select_fewshot(SRC_BY_ID[TEST_IDS[0]])],
    ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Версия промптов: {PROMPT_VERSION}")
for name, text in PROMPT_REGISTRY.items():
    print(f"  {name:<18} sha256:{prompt_hash(text)}  ({len(text)} символов)")
print("\nПример пользовательского сообщения (controlled):\n")
print(user_payload(TEST_IDS[0], SRC_BY_ID[TEST_IDS[0]], with_terms=True))

## Блок 5. JSON Schema и проверка ответа

**Зачем.** Ответ модели должен разбираться программой без ручной чистки. Схема задаёт поля
`segment_id`, `translation`, `used_terms`, `warnings`; модель Pydantic проверяет их типы;
дополнительные правила ловят то, что схема пропускает.

**На что смотреть.** Два контрольных примера в выводе: корректный ответ проходит, ответ без
обязательных полей отклоняется с понятной причиной.

In [ ]:
# ============================================================================
#  Блок 5. JSON Schema, модель Pydantic и валидация ответа
# ============================================================================
from pydantic import BaseModel, Field, ValidationError

SCHEMA: Dict[str, Any] = {
    "type": "object",
    "properties": {
        "segment_id": {"type": "string"},
        "translation": {"type": "string"},
        "used_terms": {
            "type": "array",
            "items": {"type": "object",
                      "properties": {"source": {"type": "string"}, "target": {"type": "string"}},
                      "required": ["source", "target"],
                      "additionalProperties": False},
        },
        "warnings": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["segment_id", "translation", "used_terms", "warnings"],
    "additionalProperties": False,
}
(PROMPTS / "translation_schema.json").write_text(json.dumps(SCHEMA, ensure_ascii=False, indent=2),
                                                 encoding="utf-8")


class UsedTerm(BaseModel):
    source: str
    target: str


class TranslationResult(BaseModel):
    """Все четыре поля обязательны — так же, как в JSON Schema (required)."""
    segment_id: str
    translation: str
    used_terms: List[UsedTerm]
    warnings: List[str]


def cyrillic_share(text: str) -> float:
    letters = re.findall(r"[A-Za-zА-Яа-яЁё]", text)
    return sum(1 for ch in letters if re.match(r"[А-Яа-яЁё]", ch)) / max(len(letters), 1)


def validate_response(raw: str, segment_id: str) -> Tuple[Optional[TranslationResult], List[str]]:
    """Три уровня проверки: синтаксис JSON → схема Pydantic → смысловые правила."""
    try:
        data = json.loads(raw)
    except json.JSONDecodeError as error:
        return None, [f"JSON: {error.msg}"]
    try:
        parsed = TranslationResult.model_validate(data)
    except ValidationError as error:
        return None, [f"схема: {str(error).splitlines()[0]}"]
    problems = []
    # TODO 5.1: смысловые проверки, которые JSON Schema не ловит:
    #   * segment_id в ответе совпадает с запрошенным;
    #   * перевод не пустой;
    #   * перевод на русском: cyrillic_share(parsed.translation) >= 0.5
    #     (вспомните NLLB без forced_bos_token_id в ЛР № 2).
    # Каждое нарушение добавляйте в problems строкой с понятной причиной.
    return (parsed if not problems else None), problems


print("Схема сохранена:", PROMPTS / "translation_schema.json")
checks = {
    "корректный ответ": '{"segment_id": "s001", "translation": "Проверка связи.", "used_terms": [], "warnings": []}',
    "нет обязательных полей": '{"segment_id": "s001", "translation": "Проверка связи."}',
    "перевод не на русском": '{"segment_id": "s001", "translation": "La búsqueda.", "used_terms": [], "warnings": []}',
    "обрезанный JSON": '{"segment_id": "s001", "translation": "Провер',
}
for label, raw in checks.items():
    parsed, problems = validate_response(raw, "s001")
    print(f"{label:<24} → {'принят' if parsed else 'отклонён: ' + '; '.join(problems)}")

## Блок 6. Единый вызов LLM и трассировка

**Зачем.** Код режимов не должен зависеть от провайдера: одна функция `translate()` строит
сообщения, вызывает выбранную модель, проверяет ответ, при ошибке повторяет запрос с
указанием проблемы и возвращает запись трассировки.

**Как устроены режимы.** zero — короткая инструкция без терминов; controlled — системная
инструкция и термины сегмента; few-shot — то же плюс примеры в виде пар «запрос → JSON-ответ»:
так модель видит не только перевод, но и точный формат.

**На что смотреть.** Пробный вызов на первом сегменте: `json_valid: true`, число попыток,
время ответа и версия модели.

In [ ]:
# ============================================================================
#  Блок 6. Единый вызов LLM: Ollama, GigaChat, Yandex AI Studio (+ mock для проверки)
# ============================================================================
def _call_ollama(messages: List[Dict[str, str]]) -> Tuple[str, Dict[str, Any]]:
    from ollama import chat
    response = chat(model=MODEL, messages=messages, format=SCHEMA,
                    options={"temperature": TEMPERATURE, "seed": SEED,
                             "num_predict": MAX_TOKENS})
    return response.message.content, {"model_version": OLLAMA_DIGEST}


def _call_gigachat(messages: List[Dict[str, str]]) -> Tuple[str, Dict[str, Any]]:
    from gigachat import GigaChat
    from gigachat.models import Chat, Messages, MessagesRole
    roles = {"system": MessagesRole.SYSTEM, "user": MessagesRole.USER,
             "assistant": MessagesRole.ASSISTANT}
    chat_request = Chat(
        model=MODEL,
        messages=[Messages(role=roles[m["role"]], content=m["content"]) for m in messages],
        temperature=max(TEMPERATURE, 0.01),  # GigaChat ожидает положительную температуру
        max_tokens=MAX_TOKENS,
        response_format={"type": "json_schema", "schema": SCHEMA, "strict": True},
    )
    with GigaChat(credentials=get_secret("GIGACHAT_CREDENTIALS"),
                  scope=get_secret("GIGACHAT_SCOPE") or "GIGACHAT_API_PERS",
                  verify_ssl_certs=(get_secret("GIGACHAT_VERIFY_SSL") or "false") == "true") as client:
        response = client.chat(chat_request)
    return response.choices[0].message.content, {"model_version": getattr(response, "model", None)}


def _call_yandex(messages: List[Dict[str, str]]) -> Tuple[str, Dict[str, Any]]:
    try:
        from yandex_ai_studio_sdk import AIStudio as SDK
    except ImportError:                         # прежнее название библиотеки
        from yandex_cloud_ml_sdk import YCloudML as SDK
    sdk = SDK(folder_id=get_secret("YANDEX_FOLDER_ID"), auth=get_secret("YANDEX_API_KEY"))
    model = sdk.models.completions(MODEL, model_version="latest").configure(
        temperature=TEMPERATURE, max_tokens=MAX_TOKENS, response_format={"json_schema": SCHEMA})
    result = model.run([{"role": m["role"], "text": m["content"]} for m in messages])
    return result[0].text, {"model_version": getattr(result, "model_version", None)}


def _call_mock(messages: List[Dict[str, str]]) -> Tuple[str, Dict[str, Any]]:
    """НЕ модель. Эхо эталона ЛР № 1 для проверки кода; в zero-режиме «забывает» первый термин."""
    last = messages[-1]["content"]
    sid = re.search(r"segment_id:\s*(\S+)", last).group(1)
    source = re.search(r"<<<\n(.*)\n>>>", last, re.S)
    translation = REF_BY_ID.get(sid, "Перевод.")
    if source and source.group(1) != SRC_BY_ID.get(sid):     # размеченный вариант сегмента
        translation = MARKUP_EXPECTED.get(sid, translation) if "MARKUP_EXPECTED" in globals() else translation
    if messages[0]["content"] == ZERO_SYSTEM:
        terms = relevant_terms(SRC_BY_ID.get(sid, ""))
        if terms:
            translation = re.sub(re.escape(terms[0][1]), terms[0][0], translation, flags=re.I, count=1)
    return json.dumps({"segment_id": sid, "translation": translation, "used_terms": [],
                       "warnings": ["MOCK: не вывод модели"]}, ensure_ascii=False), {"model_version": "mock"}


CALLERS = {"ollama": _call_ollama, "gigachat": _call_gigachat,
           "yandex": _call_yandex, "mock": _call_mock}


def build_messages(segment_id: str, source: str, mode: str, style: str = None) -> List[Dict[str, str]]:
    """zero — короткая инструкция; controlled — системный промпт + термины;
    fewshot — то же + примеры из пула в виде пар user/assistant."""
    if mode == "zero":
        return [{"role": "system", "content": ZERO_SYSTEM},
                {"role": "user", "content": user_payload(segment_id, source, with_terms=False)}]
    system = PROMPT_REGISTRY.get(f"system_style_{style}", SYSTEM_PROMPT) if style else SYSTEM_PROMPT
    messages = [{"role": "system", "content": system}]
    if mode == "fewshot":
        for sid in select_fewshot(source):
            messages.append({"role": "user", "content": user_payload(sid, SRC_BY_ID[sid], True)})
            messages.append({"role": "assistant", "content": json.dumps(
                {"segment_id": sid, "translation": REF_BY_ID[sid],
                 "used_terms": [{"source": en, "target": ru} for en, ru in relevant_terms(SRC_BY_ID[sid])],
                 "warnings": []}, ensure_ascii=False)})
    messages.append({"role": "user", "content": user_payload(segment_id, source, with_terms=True)})
    return messages


def translate(segment_id: str, source: str, mode: str, style: str = None,
              messages: List[Dict[str, str]] = None, max_attempts: int = 3) -> Dict[str, Any]:
    """Вызов с повтором при невалидном JSON. Возвращает запись трассировки."""
    messages = messages or build_messages(segment_id, source, mode, style)
    record = {"segment_id": segment_id, "provider": PROVIDER, "model": MODEL, "mode": mode,
              "style": style, "temperature": TEMPERATURE, "prompt_version": PROMPT_VERSION,
              "prompt_hash": prompt_hash(messages[0]["content"]), "source": source,
              "translation": "", "used_terms": [], "warnings": [], "json_valid": False,
              "attempts": 0, "errors": [], "model_version": None, "latency_s": None}
    for attempt in range(1, max_attempts + 1):
        started = time.perf_counter()
        try:
            raw, meta = CALLERS[PROVIDER](messages)
        except Exception as error:                          # сеть, авторизация, квоты
            record["errors"].append(f"вызов: {type(error).__name__}: {error}"[:300])
            break
        record.update(attempts=attempt, latency_s=round(time.perf_counter() - started, 2),
                      model_version=meta.get("model_version"))
        parsed, problems = validate_response(raw, segment_id)
        if parsed:
            record.update(translation=parsed.translation, json_valid=True,
                          used_terms=[t.model_dump() for t in parsed.used_terms],
                          warnings=parsed.warnings)
            break
        record["errors"].extend(problems)
        messages = messages + [{"role": "user", "content":
                                "Ответ не прошёл проверку: " + "; ".join(problems) +
                                ". Верни только JSON строго по схеме."}]
    return record


probe = translate(TEST_IDS[0], SRC_BY_ID[TEST_IDS[0]], "controlled")
print(json.dumps({k: probe[k] for k in ("segment_id", "mode", "translation", "json_valid",
                                         "attempts", "latency_s", "model_version", "errors")},
                 ensure_ascii=False, indent=2))

## Блок 7. Три режима на одинаковых сегментах

**Зачем.** Основной эксперимент работы: одни и те же 12 сегментов переводятся тремя способами,
все вызовы записываются в `results_llm.jsonl` с полной трассировкой.

**На что смотреть.** Сводку по режимам: долю валидных JSON-ответов, среднее число попыток и время
ответа. Если в каком-то режиме валидность ниже 100 %, посмотрите поле `errors` — обычно это
выход за формат или ответ не на том языке.

In [ ]:
# ============================================================================
#  Блок 7. Три режима на одних и тех же сегментах: zero / controlled / few-shot
# ============================================================================
MODES = ["zero", "controlled", "fewshot"]
records: List[Dict[str, Any]] = []
for mode in MODES:
    for sid in TEST_IDS:
        records.append(translate(sid, SRC_BY_ID[sid], mode))
    print(f"{mode:<11} готово: {sum(r['mode'] == mode for r in records)} сегментов")

with open(RESULTS / "results_llm.jsonl", "w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

runs = pd.DataFrame(records)
summary = (runs.groupby("mode", sort=False)
           .agg(segments=("segment_id", "count"), json_valid=("json_valid", "mean"),
                mean_attempts=("attempts", "mean"), mean_latency_s=("latency_s", "mean"),
                with_warnings=("warnings", lambda s: sum(bool(w) for w in s)))
           .reindex(MODES))
summary["json_valid"] = (summary["json_valid"] * 100).round(1)
print("\nСводка по режимам (json_valid — доля валидных ответов, %):")
summary

## Блок 8. Терминологический контроль

**Зачем.** Главный вопрос работы — помогает ли управление соблюдать глоссарий. Здесь число
нарушений считается для каждого режима LLM и, если подключены результаты ЛР № 2, для MarianMT и
NLLB — на одних и тех же сегментах и по одной процедуре.

**Что учтено из ЛР № 2.** В списке окончаний добавлено «-ую»: без него «наземную станцию» не
совпадала с «наземная станция», и проверка давала ложное нарушение.

**На что смотреть.** Долю соблюдения по системам. Ожидаемая картина: zero-shot близок к NMT,
controlled и few-shot заметно выше. Колонка `confirmed` в `terminology_qa.csv` заполняется
вручную: `yes` — нарушение подтверждено, `no` — ложное срабатывание.

In [ ]:
# ============================================================================
#  Блок 8. Терминологический контроль: все режимы LLM и (при наличии) NMT из ЛР № 2
# ============================================================================
RU_ENDINGS = sorted({"иями", "ями", "ами", "ого", "его", "ому", "ему", "ыми", "ими", "ую", "юю",
                     "ая", "яя", "ое", "ее", "ые", "ие", "ый", "ий", "ой", "ей", "ом", "ем", "ём",
                     "ах", "ях", "ам", "ям", "ов", "ев", "ым", "им", "ых", "их", "ь", "и", "ы",
                     "а", "я", "е", "о", "у", "ю"}, key=len, reverse=True)


def ru_stem(word: str) -> str:
    """Грубая нормализация: отсечение окончания. В ЛР № 2 в списке не было «-ую», из-за чего
    «наземную станцию» не совпадала с «наземная станция» — здесь список дополнен."""
    word = word.lower().replace("ё", "е").strip(".,;:()«»\"'`*")
    for ending in RU_ENDINGS:
        if len(word) - len(ending) >= 3 and word.endswith(ending):
            return word[: -len(ending)]
    return word


def token_match(first: str, second: str) -> bool:
    """Совпадение двух слов с учётом словоизменения.

    1) одинаковые основы («наземную» — «наземная», «шага» — «шаг»);
    2) беглая гласная в коротких словах: общее начало и высокое сходство («угол» — «угла»).
    Прилагательное и существительное одного корня («полётный» — «полёта») не совпадают:
    для глоссария это разные решения.
    """
    first, second = ru_stem(first), ru_stem(second)
    if first == second:
        return True
    return (max(len(first), len(second)) <= 4 and first[:2] == second[:2]
            and difflib.SequenceMatcher(None, first, second).ratio() >= 0.75)


def term_present(term_ru: str, text: str) -> bool:
    tokens = re.findall(r"[\wёЁ-]+", text.lower())
    return all(any(token_match(part, token) for token in tokens)
               for part in re.findall(r"[\wёЁ-]+", term_ru.lower()))


def term_violations(source: str, translation: str) -> List[Dict[str, str]]:
    return [{"source_term": en, "expected": ru} for en, ru in relevant_terms(source)
            if not term_present(ru, translation)]


systems = {mode: {r["segment_id"]: r["translation"] for r in records if r["mode"] == mode}
           for mode in MODES}
for name, table in NMT.items():
    if table:
        systems[name] = {sid: table.get(sid, "") for sid in TEST_IDS}

qa_rows, compliance = [], []
occurrences = sum(len(relevant_terms(SRC_BY_ID[sid])) for sid in TEST_IDS)
for system, table in systems.items():
    violations = 0
    for sid in TEST_IDS:
        for violation in term_violations(SRC_BY_ID[sid], table.get(sid, "")):
            violations += 1
            qa_rows.append({"segment_id": sid, "system": system, **violation,
                            "translation": table.get(sid, ""), "confirmed": "", "comment": ""})
    compliance.append({"system": system, "occurrences": occurrences, "violations": violations,
                       "compliance_%": round(100 * (1 - violations / max(occurrences, 1)), 1)})

terminology_qa = pd.DataFrame(qa_rows, columns=["segment_id", "system", "source_term", "expected",
                                                "translation", "confirmed", "comment"])
terminology_qa.to_csv(RESULTS / "terminology_qa.csv", index=False, encoding="utf-8-sig")
compliance_df = pd.DataFrame(compliance)
print(f"Вхождений терминов в тестовые сегменты: {occurrences}")
compliance_df

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(8, 3.8))
colors = ["#9e9e9e" if s in ("MarianMT", "NLLB") else "#1565c0" for s in compliance_df["system"]]
axis.barh(compliance_df["system"], compliance_df["compliance_%"], color=colors)
for y, value in enumerate(compliance_df["compliance_%"]):
    axis.text(value + 1, y, f"{value:.1f} %", va="center")
axis.set_xlim(0, 110)
axis.invert_yaxis()
axis.set_xlabel("Соблюдение глоссария до ручной проверки, %")
axis.set_title(f"Терминология: {occurrences} вхождений терминов в {len(TEST_IDS)} сегментах")
axis.grid(axis="x", alpha=0.3)
figure.tight_layout()
figure.savefig(FIGURES / "terminology_compliance.png", dpi=150)
plt.show()

## Блок 9. Разметка, числа, код и модальность

**Зачем.** В учебном тексте часть элементов нельзя переводить или менять. Проверочные случаи
строятся из ваших сегментов: выделение термина `**…**`, список из двух сегментов и фрагмент кода
с числом и путём к файлу.

**На что смотреть.** Четыре флага: выделение, список, код и числа сохранены. Отдельная таблица
показывает, передана ли модальность (*must*, *should*, *require*) в сегментах, где она есть.

In [ ]:
# ============================================================================
#  Блок 9. Сохранение разметки, чисел, кода и модальности
#  Исходные сегменты — обычный текст, поэтому проверочные случаи строятся из них:
#  выделение термина, список из двух сегментов, фрагмент кода и числа с единицами.
# ============================================================================
def markup_cases() -> List[Dict[str, str]]:
    with_terms = [sid for sid in TEST_IDS if relevant_terms(SRC_BY_ID[sid])]
    base = (with_terms + [sid for sid in TEST_IDS if sid not in with_terms])[:3]
    base = (base * 3)[:3]                       # на случай очень маленького корпуса
    cases = []
    sid = base[0]
    terms = relevant_terms(SRC_BY_ID[sid])
    term_en, term_ru = terms[0] if terms else (SRC_BY_ID[sid].split()[1], "—")
    cases.append({"case": "bold", "segment_id": sid,
                  "source": re.sub(re.escape(term_en), f"**{term_en}**", SRC_BY_ID[sid], count=1,
                                   flags=re.I),
                  "expected": f"термин «{term_ru}» выделен **…**"})
    first, second = base[1], base[2]
    cases.append({"case": "list", "segment_id": first,
                  "source": f"- {SRC_BY_ID[first]}\n- {SRC_BY_ID[second]}",
                  "expected": "два пункта списка, маркеры «- » сохранены"})
    cases.append({"case": "code", "segment_id": base[0],
                  "source": SRC_BY_ID[base[0]] + " Set `MAX_RATE = 200` in `config/params.yaml`.",
                  "expected": "фрагменты в `…` не изменены"})
    return cases


def markup_report(source: str, translation: str) -> Dict[str, Any]:
    code_src, code_tgt = re.findall(r"`[^`]+`", source), re.findall(r"`[^`]+`", translation)
    nums_src, nums_tgt = re.findall(r"\d+(?:[.,]\d+)?", source), re.findall(r"\d+(?:[.,]\d+)?", translation)
    return {"bold_ok": source.count("**") == translation.count("**"),
            "list_ok": len(re.findall(r"^- ", source, re.M)) == len(re.findall(r"^- ", translation, re.M)),
            "code_ok": code_src == code_tgt,
            "numbers_ok": sorted(n.replace(",", ".") for n in nums_src)
                          == sorted(n.replace(",", ".") for n in nums_tgt)}


MODAL_EN = r"\b(must|should|can|cannot|may|need to|requires?|required)\b"
MODAL_RU = r"(должн|следует|может|могут|можно|нельзя|необходим|нужно|требу)"

MARKUP_EXPECTED = {}                       # используется только mock-режимом
markup_rows = []
for case in markup_cases():
    record = translate(case["segment_id"], case["source"], "controlled")
    record["mode"] = f"markup_{case['case']}"
    records.append(record)
    markup_rows.append({"case": case["case"], "segment_id": case["segment_id"],
                        "source": case["source"], "translation": record["translation"],
                        **markup_report(case["source"], record["translation"])})
markup_df = pd.DataFrame(markup_rows)
markup_df.to_csv(RESULTS / "markup_checks.csv", index=False, encoding="utf-8-sig")

modality = []
for sid in TEST_IDS:
    if re.search(MODAL_EN, SRC_BY_ID[sid], re.I):
        for mode in MODES:
            modality.append({"segment_id": sid, "mode": mode,
                             "modal_en": re.search(MODAL_EN, SRC_BY_ID[sid], re.I).group(0),
                             "preserved": bool(re.search(MODAL_RU, systems[mode][sid], re.I)),
                             "translation": systems[mode][sid]})
modality_df = pd.DataFrame(modality)
print("Проверка разметки (режим controlled):")
print(markup_df[["case", "bold_ok", "list_ok", "code_ok", "numbers_ok"]].to_string(index=False))
print(f"\nСегментов с модальностью: {modality_df['segment_id'].nunique() if len(modality_df) else 0}")
modality_df

## Блок 10. Targeted repair

**Зачем.** Показать, что нарушение можно исправить **точечно**, не переводя сегмент заново. Для
сегментов с нарушениями отправляется узкая инструкция со списком «английский термин →
обязательный эквивалент».

**На что смотреть.** Число нарушений до и после и `unchanged_ratio` — долю текста, оставшегося
прежним. Хороший repair: нарушений стало 0, `unchanged_ratio` около 0.9.

In [ ]:
# ============================================================================
#  Блок 10. Targeted repair: узкое исправление только найденных нарушений
# ============================================================================
PRIORITY = ["fewshot", "controlled", "zero"]
candidates = []
for mode in PRIORITY:
    for sid in TEST_IDS:
        found = term_violations(SRC_BY_ID[sid], systems[mode][sid])
        if found:
            candidates.append((mode, sid, found))
print(f"Сегментов с нарушениями: {len(candidates)} (минимум для работы: {N_REPAIR_MIN})")

repair_rows = []
# TODO 10.1: для первых max(N_REPAIR_MIN, 5) кандидатов:
#   1) соберите промпт REPAIR_TEMPLATE.format(...) со списком нарушений;
#   2) вызовите translate(sid, source, f"repair_{mode}", messages=[system, user]);
#   3) повторно выполните term_violations для исправленного текста;
#   4) посчитайте долю неизменённого текста difflib.SequenceMatcher(None, before, after).ratio();
#   5) добавьте строку в repair_rows с полями segment_id, mode, violations_before,
#      violations_after, unchanged_ratio, before, after, terms.

repair_df = pd.DataFrame(repair_rows)
repair_df.to_csv(RESULTS / "repair_cases.csv", index=False, encoding="utf-8-sig")
repair_df

## Блок 11. Управление стилем

**Зачем.** Один и тот же сегмент переводится в академическом стиле (A) и в стиле преподавателя,
объясняющего первокурснику (B). Для педагога это ключевая возможность LLM: текст можно адаптировать
под аудиторию, не меняя терминологию.

**На что смотреть.** Простые признаки стиля: обращения на «вы», побудительные формы («проверьте»),
безличные и пассивные конструкции («калибруется»). Стиль B должен давать обращения и
побуждения, стиль A — безличные конструкции; термины в обоих должны совпадать.

In [ ]:
# ============================================================================
#  Блок 11. Управление стилем: академический (A) и инструктивный (B)
# ============================================================================
def style_metrics(text: str) -> Dict[str, Any]:
    tokens = re.findall(r"[\wёЁ-]+", text)
    return {"words": len(tokens),
            "you_forms": len(re.findall(r"\b(вы|вам|вас|ваш\w*)\b", text, re.I)),
            "imperatives": len(re.findall(r"\b\w+(?:ите|йте)\b", text, re.I)),
            "passive_or_impersonal": len(re.findall(r"\b\w+(?:ется|ются|ится|ятся)\b", text, re.I))}


style_ids = [sid for sid in TEST_IDS if re.search(r"\b(operator|before|check|upload|calibrat)", SRC_BY_ID[sid], re.I)]
style_ids = (style_ids + [sid for sid in TEST_IDS if sid not in style_ids])[:N_STYLE]
style_rows = []
for sid in style_ids:
    row = {"segment_id": sid, "source": SRC_BY_ID[sid]}
    for style in ("A", "B"):
        record = translate(sid, SRC_BY_ID[sid], f"style_{style}", style=style)
        records.append(record)
        row[f"style_{style}"] = record["translation"]
        row.update({f"{k}_{style}": v for k, v in style_metrics(record["translation"]).items()})
    style_rows.append(row)
style_df = pd.DataFrame(style_rows)
style_df.to_csv(RESULTS / "style_comparison.csv", index=False, encoding="utf-8-sig")
for row in style_rows:
    print(f"[{row['segment_id']}] {row['source']}\n  A: {row['style_A']}\n  B: {row['style_B']}\n")
style_df.drop(columns=["source", "style_A", "style_B"])

## Блок 12. Сводная таблица и экспертные решения

**Зачем.** Свести эталон ЛР № 1, переводы NMT из ЛР № 2 и три режима LLM в одну таблицу и принять
по сегментам экспертные решения. Типы различий — как в ЛР № 2: точность, терминология, беглость,
стиль, разметка.

**Что заполнить вручную.** Словарь `EXPERT` (минимум пять сегментов): кто предпочтительнее и
почему.

In [ ]:
# ============================================================================
#  Блок 12. Сводная таблица и экспертные решения
# ============================================================================
comparison = pd.DataFrame({"segment_id": TEST_IDS,
                           "source": [SRC_BY_ID[s] for s in TEST_IDS],
                           "reference": [REF_BY_ID[s] for s in TEST_IDS]})
for system, table in systems.items():
    comparison[system] = [table.get(sid, "") for sid in TEST_IDS]
comparison["preferred"] = ""
comparison["comment"] = ""

# Экспертные решения заполняются ПОСЛЕ прочтения фактических переводов выше.
# Значение выбора: "zero", "controlled", "fewshot", "MarianMT", "NLLB" или "ни один";
# обоснование указывает тип различия (точность / терминология / беглость / стиль / разметка).
EXPERT: Dict[str, Tuple[str, str]] = {
    # "s001": ("fewshot", "Терминология: ..."),
}
for sid, (choice, note) in EXPERT.items():
    comparison.loc[comparison["segment_id"] == sid, ["preferred", "comment"]] = [choice, note]
comparison.to_csv(RESULTS / "comparison.csv", index=False, encoding="utf-8-sig")

for row in comparison.head(6).to_dict("records"):
    print(f"[{row['segment_id']}] {row['source']}")
    print(f"  эталон     : {row['reference']}")
    for system in systems:
        print(f"  {system:<11}: {row[system]}")
    print()
print(f"Заполнено экспертных решений: {int((comparison['preferred'] != '').sum())} (требуется >= 5)")

## Автотесты

Проверяют ваши функции без вызова LLM. Все строки должны быть отмечены ✅.

In [ ]:
# ============================================================================
#  Автотесты: проверяют ваши функции без вызова LLM
# ============================================================================
def test_relevant_terms():
    sample = "The operator uploads a new waypoint to the autopilot."
    terms = dict(relevant_terms(sample, pd.DataFrame({"en": ["waypoint", "autopilot", "airframe"],
                                                      "ru": ["путевая точка", "автопилот", "планёр"]})))
    assert set(terms) == {"waypoint", "autopilot"}, terms


def test_validate_response():
    ok, problems = validate_response('{"segment_id": "s001", "translation": "Проверка связи.", '
                                     '"used_terms": [], "warnings": []}', "s001")
    assert ok is not None and not problems
    bad, problems = validate_response('{"segment_id": "s002", "translation": "Проверка.", '
                                      '"used_terms": [], "warnings": []}', "s001")
    assert bad is None and problems
    bad, problems = validate_response('{"segment_id": "s001", "translation": "La búsqueda.", '
                                      '"used_terms": [], "warnings": []}', "s001")
    assert bad is None, "перевод не на русском должен отклоняться"


def test_term_present():
    assert term_present("наземная станция управления", "Система включает наземную станцию управления.")
    assert term_present("угол атаки", "Превышение критического угла атаки приводит к сваливанию.")
    assert term_present("шаг воздушного винта", "Увеличение шага воздушного винта снижает обороты.")
    assert not term_present("планёр", "Регулятор стабилизирует воздушное пространство.")


def test_markup_report():
    report = markup_report("Set `MAX_RATE = 200` in **flight**.", "Задайте `MAX_RATE = 200` в **полёте**.")
    assert all(report.values()), report


def test_no_leakage():
    assert not set(TEST_IDS) & set(POOL_IDS)


for test in (test_relevant_terms, test_validate_response, test_term_present,
             test_markup_report, test_no_leakage):
    try:
        test()
        print(f"✅ {test.__name__}")
    except NameError as error:
        print(f"⏭ {test.__name__}: функция ещё не определена ({error})")
    except AssertionError as error:
        print(f"❌ {test.__name__}: {error}")

## Блок 13. Отчёт, версии промптов и выгрузка

**Зачем.** Собрать репозиторий по требованиям работы: `report.md`, `prompt_versions.md`,
`requirements.txt`, `.env.example`, `.gitignore`, результаты и промпты. В `.env.example` — только
имена переменных, без значений.

In [ ]:
# ============================================================================
#  Блок 13. Отчёт, версии промптов, зависимости и выгрузка
# ============================================================================
import zipfile

prompt_table = "\n".join(f"| `{name}` | `{prompt_hash(text)}` | {len(text)} |"
                         for name, text in PROMPT_REGISTRY.items())
(BASE / "prompt_versions.md").write_text(
    f"# Версии промптов\n\nВерсия набора: **{PROMPT_VERSION}**\n\n"
    f"| Промпт | sha256 (10 знаков) | Длина |\n|---|---|---:|\n{prompt_table}\n", encoding="utf-8")

(BASE / "requirements.txt").write_text(
    "\n".join(["pydantic>=2", "python-dotenv", f"pandas=={pd.__version__}",
               {"ollama": "ollama", "gigachat": "gigachat",
                "yandex": "yandex-ai-studio-sdk"}.get(PROVIDER, "")]) + "\n", encoding="utf-8")
(BASE / ".env.example").write_text(
    "GIGACHAT_CREDENTIALS=\nGIGACHAT_SCOPE=GIGACHAT_API_PERS\nYANDEX_FOLDER_ID=\nYANDEX_API_KEY=\n",
    encoding="utf-8")
(BASE / ".gitignore").write_text(".env\n.venv/\n__pycache__/\n*.pyc\n", encoding="utf-8")

with open(RESULTS / "results_llm.jsonl", "w", encoding="utf-8") as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")

valid_share = runs.groupby("mode")["json_valid"].mean().mul(100).round(1).to_dict()
compliance_lines = "\n".join(f"| {r['system']} | {r['violations']} | {r['compliance_%']} % |"
                             for r in compliance_df.to_dict("records"))
expert_rows = comparison.loc[comparison["preferred"] != "", ["segment_id", "preferred", "comment"]]
report = f"""# Отчёт по лабораторной работе 3 (вариант {VARIANT})

**Предметная область (сквозной кейс ЛР № 1–2):** {VARIANT_DOMAIN or DOMAIN_EN}

## 1–3. Модель, версия, авторизация
- Провайдер: `{PROVIDER}`, модель: `{MODEL}`, версия/сборка: `{ENVIRONMENT.get('model_digest') or records[0].get('model_version')}`
- Параметры: temperature={TEMPERATURE}, seed={SEED}, max_tokens={MAX_TOKENS}
- Авторизация: ключи читаются из Colab Secrets / `.env` (в репозитории только `.env.example`)
{"- ⚠ Режим mock: модель не вызывалась, работа к сдаче не принимается" if PROVIDER == "mock" else ""}

## 4–5. Системный промпт и JSON Schema
Версия промптов {PROMPT_VERSION}; файлы `prompts/`, хэши — `prompt_versions.md`.
Схема: `prompts/translation_schema.json` (segment_id, translation, used_terms, warnings).

## 6. Режимы zero / controlled / few-shot ({len(TEST_IDS)} сегментов)
Доля валидных JSON-ответов, %: {valid_share}

| Система | Нарушений глоссария | Соблюдение |
|---|---:|---:|
{compliance_lines}

## 7. Анализ ошибок
Экспертных решений: {len(expert_rows)}
""" + "\n".join(f"- {r.segment_id}: {r.preferred} — {r.comment}" for r in expert_rows.itertuples(index=False)) + f"""

## 8. Terminology repair
Случаев: {len(repair_df)}; нарушений до/после: {int(repair_df['violations_before'].sum()) if len(repair_df) else 0} / {int(repair_df['violations_after'].sum()) if len(repair_df) else 0}.

## 9. Стиль A / B
Сегментов: {len(style_df)}; см. `results/style_comparison.csv`.

## 10. Ограничения и вывод
Разметка сохранена во всех проверочных случаях: {bool(markup_df[['bold_ok', 'list_ok', 'code_ok', 'numbers_ok']].all().all())}.
Ограничения: {len(TEST_IDS)} сегментов, одна модель, temperature={TEMPERATURE}; автоматическая QA требует ручного подтверждения.
"""
(BASE / "report.md").write_text(report, encoding="utf-8")

archive = shutil.make_archive(str(BASE.parent / "lab03_results"), "zip",
                              root_dir=BASE.parent, base_dir=BASE.name)
print("Созданные файлы:")
for path in sorted(BASE.rglob("*")):
    if path.is_file() and "lab03_results" not in path.name:
        print(f"  {str(path.relative_to(BASE)):<42} {path.stat().st_size:>8} байт")
print("Архив:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass

## Литература и источники для углублённого изучения

### LLM и перевод

1. ▸ **Vilar D., Freitag M., Cherry C. et al.** Prompting PaLM for Translation: Assessing Strategies and Performance // ACL. — 2023. — arXiv:2211.09102. — Как выбор примеров влияет на качество перевода LLM.
2. **Hendy A. et al.** How Good Are GPT Models at Machine Translation? A Comprehensive Evaluation. — 2023. — arXiv:2302.09210.
3. **Zhu W. et al.** Multilingual Machine Translation with Large Language Models: Empirical Results and Analysis // Findings of NAACL. — 2024. — arXiv:2304.04675.
4. **Brown T. et al.** Language Models are Few-Shot Learners // NeurIPS. — 2020. — arXiv:2005.14165. — Обучение по примерам в запросе.

### Терминология, память переводов и примеры

5. ▸ **Moslem Y., Haque R., Kelleher J., Way A.** Adaptive Machine Translation with Large Language Models // EAMT. — 2023. — arXiv:2301.13294. — Нечёткие совпадения из памяти переводов как примеры few-shot: прямое продолжение ЛР № 1.
6. ▸ **Ghazvininejad M., Gonen H., Zettlemoyer L.** Dictionary-based Phrase-level Prompting of Large Language Models for Machine Translation. — 2023. — arXiv:2302.07856. — Передача в промпт только релевантных словарных пар.
7. **Dinu G. et al.** Training Neural Machine Translation to Apply Terminology Constraints // ACL. — 2019. — arXiv:1906.01105. — Терминологические ограничения в NMT: с чем сравнивать подход LLM.

### Структурированный вывод и оценка

8. **Willard B., Louf R.** Efficient Guided Generation for Large Language Models. — 2023. — arXiv:2307.09702. — Как генерация ограничивается схемой.
9. **Kocmi T., Federmann C.** Large Language Models Are State-of-the-Art Evaluators of Translation Quality // EAMT. — 2023. — arXiv:2302.14520. — Задел для ЛР № 4.
10. **Lommel A., Uszkoreit H., Burchardt A.** Multidimensional Quality Metrics (MQM) // Tradumàtica. — 2014. — Типология ошибок для экспертного разбора.
11. **ISO 18587:2017.** Translation services — Post-editing of machine translation output — Requirements.

### Документация

12. GigaChat: генерация структурированных данных. — <https://developers.sber.ru/docs/ru/gigachat/guides/structured-output>
13. Yandex AI Studio: структурированный вывод. — <https://yandex.cloud/ru/docs/ai-studio/concepts/generation/structured-output>
14. Ollama: structured outputs. — <https://ollama.com/blog/structured-outputs>
15. Pydantic: модели и валидация. — <https://docs.pydantic.dev/>

**С чего начать:** 1, 5 и 6 — они напрямую объясняют решения, принятые в этой работе.

## Блок 14. Выводы

*(6–8 пунктов по вашим результатам: режимы, JSON, разметка и модальность, repair, стиль, сравнение
с NMT из ЛР № 2, ограничения эксперимента)*

1. …
2. …